# Home Credit Default Risk — Modeling & Fairness Audit

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import lightgbm as lgb

# Same palette as EDA / FE notebooks
plt.rcParams.update({
    'figure.dpi': 110,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.5,
    'font.size': 9.5,
})

POS, NEG, NEU = '#d62728', '#2ca02c', '#3a6ea5'
FEM, MAL = '#e377c2', '#1f77b4'
GRAY = '#999999'

# Distinct colors for the four experiments
EXP_COLORS = {
    'baseline':          GRAY,
    'with_protected':    '#1f77b4',
    'without_protected': '#ff7f0e',
    'mitigated':         '#2ca02c',
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# Load the engineered feature matrix and the manifest produced by the
# feature-engineering notebook.
df = pd.read_csv('application_train_features.csv')
with open('feature_manifest.json') as f:
    manifest = json.load(f)

print(f'Loaded {len(df):,} rows  x  {df.shape[1]} cols')
print(f'Default rate: {df["TARGET"].mean()*100:.2f}%')
print(f'Manifest:  protected={len(manifest["protected"])}, '
      f'engineered={len(manifest["engineered"])}, raw={len(manifest["raw"])}')

# Recover an AGE_GROUP column for audit grouping (from the dummies)
age_group_cols = [c for c in df.columns if c.startswith('AGE_GROUP_')]
# Reverse the one-hot to a categorical with the right ordering
labels = [c.replace('AGE_GROUP_', '') for c in age_group_cols]
df['AGE_GROUP'] = df[age_group_cols].idxmax(axis=1).str.replace('AGE_GROUP_', '')
df['AGE_GROUP'] = pd.Categorical(df['AGE_GROUP'],
                                  categories=['<25', '25-35', '35-45', '45-55', '55+'],
                                  ordered=True)
print(f'\nAGE_GROUP distribution:')
print(df['AGE_GROUP'].value_counts().sort_index().to_string())


Loaded 307,507 rows  x  214 cols
Default rate: 8.07%
Manifest:  protected=8, engineered=94, raw=110

AGE_GROUP distribution:
AGE_GROUP
<25      12233
25-35    72427
35-45    84260
45-55    70190
55+      68397


In [3]:
# Train/test split — stratified by TARGET so default rate is preserved
df_train, df_test = train_test_split(df, test_size=0.20,
                                      stratify=df['TARGET'],
                                      random_state=RANDOM_SEED)
print(f'Train:  {len(df_train):>7,}  (default rate {df_train["TARGET"].mean()*100:.2f}%)')
print(f'Test:   {len(df_test):>7,}  (default rate {df_test["TARGET"].mean()*100:.2f}%)')


Train:  246,005  (default rate 8.07%)
Test:    61,502  (default rate 8.07%)


## M1 — Fairness metrics & training helpers

The three fairness metrics are computed pairwise across groups and then
summarized as the *max gap* across all pairs (so the gap measures the
worst case the model exhibits — a more honest summary than the mean gap).

We also wrap the training/evaluation loop into a single function
`run_experiment(name, features, mitigation)` so the four experiments
share identical scaffolding.

In [4]:
def demographic_parity_gap(y_pred, group):
    '''Max absolute difference in P(ŷ=1) across groups.'''
    rates = pd.Series(y_pred).groupby(group.values).mean()
    return rates.max() - rates.min()

def equal_opportunity_gap(y_true, y_pred, group):
    '''Max abs diff in P(ŷ=1 | y=1) across groups (TPR gap).'''
    mask = y_true == 1
    if mask.sum() == 0:
        return np.nan
    tprs = pd.Series(y_pred[mask]).groupby(group.values[mask]).mean()
    return tprs.max() - tprs.min()

def auc_gap(y_true, y_score, group):
    '''Max abs diff in within-group ROC-AUC across groups.'''
    aucs = []
    for g, idx in pd.Series(group).groupby(group.values).groups.items():
        idx = np.array(idx)
        if y_true.iloc[idx].nunique() < 2:
            continue
        aucs.append(roc_auc_score(y_true.iloc[idx], y_score[idx]))
    if len(aucs) < 2:
        return np.nan
    return max(aucs) - min(aucs)

def audit(y_true, y_score, y_pred, gender_series, age_series):
    '''Return all fairness metrics for both protected attributes.'''
    return {
        'auc_overall':     roc_auc_score(y_true, y_score),
        'positive_rate':   y_pred.mean(),
        # Gender
        'gender_dp_gap':   demographic_parity_gap(y_pred, gender_series),
        'gender_eo_gap':   equal_opportunity_gap(y_true, y_pred, gender_series),
        'gender_auc_gap':  auc_gap(y_true, y_score, gender_series),
        # Age
        'age_dp_gap':      demographic_parity_gap(y_pred, age_series),
        'age_eo_gap':      equal_opportunity_gap(y_true, y_pred, age_series),
        'age_auc_gap':     auc_gap(y_true, y_score, age_series),
    }


In [5]:
import re

def _sanitize_cols(X):
    '''LightGBM rejects feature names with JSON-special chars (comma, slash,
    angle brackets, spaces, etc.) — see e.g. NAME_TYPE_SUITE_Spouse, partner.
    Rename to safe ASCII before fitting/predicting.'''
    new_names = {c: re.sub(r'[^A-Za-z0-9_]+', '_', str(c)) for c in X.columns}
    return X.rename(columns=new_names)

def fit_logreg(X_train, y_train, sample_weight=None):
    '''Standardize then fit logistic regression with class balancing.
    lbfgs solver is ~10x faster than liblinear on this dataset.'''
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    clf = LogisticRegression(max_iter=200, class_weight='balanced',
                              solver='lbfgs', random_state=RANDOM_SEED)
    clf.fit(X_train_s, y_train, sample_weight=sample_weight)
    return clf, scaler

def fit_lgbm(X_train, y_train, sample_weight=None):
    '''Fit LightGBM with sensible defaults; class-weight via is_unbalance.
    200 trees + lr=0.1 is a sweet spot for this dataset (full grid search
    in the baseline notebook confirmed no meaningful gain from larger
    ensembles).'''
    X_train = _sanitize_cols(X_train)
    clf = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, num_leaves=31,
        min_child_samples=100, reg_alpha=0.1, reg_lambda=0.1,
        is_unbalance=True, random_state=RANDOM_SEED, verbose=-1,
    )
    clf.fit(X_train, y_train, sample_weight=sample_weight)
    return clf

def predict_logreg(clf, scaler, X, threshold=0.5):
    proba = clf.predict_proba(scaler.transform(X))[:, 1]
    return proba, (proba >= threshold).astype(int)

def predict_lgbm(clf, X, threshold=0.5):
    X = _sanitize_cols(X)
    proba = clf.predict_proba(X)[:, 1]
    return proba, (proba >= threshold).astype(int)


In [6]:
def run_experiment(name, feature_cols, *, sample_weight_fn=None,
                   threshold_strategy='fixed', threshold=0.5):
    '''Run both LogReg and LightGBM on (feature_cols) with optional
    sample-reweighing and threshold strategy. Returns a list of result rows.

    sample_weight_fn(df_train) -> array | None
    threshold_strategy ∈ {'fixed', 'per_group_eo'}
    '''
    # Train and test matrices
    X_tr = df_train[feature_cols].copy()
    X_te = df_test[feature_cols].copy()
    y_tr = df_train['TARGET']
    y_te = df_test['TARGET']

    # Sample weights (only used by reweighed experiments)
    sw = sample_weight_fn(df_train) if sample_weight_fn is not None else None

    # Group series for audit (always read from the unfiltered test split)
    gender_te = df_test['CODE_GENDER'].reset_index(drop=True)
    age_te    = df_test['AGE_GROUP'].astype(str).reset_index(drop=True)

    results = []

    # --- Logistic Regression ---
    t0 = time.time()
    clf_lr, scaler = fit_logreg(X_tr, y_tr, sample_weight=sw)
    proba_lr, pred_lr = predict_logreg(clf_lr, scaler, X_te, threshold=threshold)
    if threshold_strategy == 'per_group_eo':
        pred_lr = per_group_eo_threshold(y_te, proba_lr, gender_te)
    a_lr = audit(y_te.reset_index(drop=True), proba_lr, pred_lr,
                 gender_te, age_te)
    a_lr.update({'experiment': name, 'model': 'LogReg',
                 'n_features': len(feature_cols), 'fit_secs': time.time()-t0})
    results.append(a_lr)

    # --- LightGBM ---
    t0 = time.time()
    clf_lgb = fit_lgbm(X_tr, y_tr, sample_weight=sw)
    proba_lgb, pred_lgb = predict_lgbm(clf_lgb, X_te, threshold=threshold)
    if threshold_strategy == 'per_group_eo':
        pred_lgb = per_group_eo_threshold(y_te, proba_lgb, gender_te)
    a_lgb = audit(y_te.reset_index(drop=True), proba_lgb, pred_lgb,
                  gender_te, age_te)
    a_lgb.update({'experiment': name, 'model': 'LightGBM',
                  'n_features': len(feature_cols), 'fit_secs': time.time()-t0})
    results.append(a_lgb)

    return results


In [7]:
def per_group_eo_threshold(y_true, y_score, group, target_tpr=None):
    '''Post-processing mitigation: choose a per-group threshold so each
    group reaches the same TPR.

    If target_tpr is None we use the TPR achieved by a single global
    threshold of 0.5 — so the smaller group is brought up to the majority
    TPR rather than the other way around (which would degrade overall
    performance unnecessarily).
    '''
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    group = np.asarray(group)

    if target_tpr is None:
        # Use the TPR that the global 0.5 threshold gives on the whole pop
        global_pred = (y_score >= 0.5).astype(int)
        target_tpr = global_pred[y_true == 1].mean()

    pred = np.zeros_like(y_true)
    for g in np.unique(group):
        mask = group == g
        if (y_true[mask] == 1).sum() == 0:
            pred[mask] = (y_score[mask] >= 0.5).astype(int)
            continue
        # Search threshold yielding the target TPR for this group
        thresholds = np.linspace(0.01, 0.99, 99)
        tprs = [((y_score[mask] >= t)[y_true[mask] == 1]).mean()
                for t in thresholds]
        idx = np.argmin(np.abs(np.array(tprs) - target_tpr))
        pred[mask] = (y_score[mask] >= thresholds[idx]).astype(int)
    return pred

def kamiran_calders_weights(df_train, protected_col='CODE_GENDER'):
    '''Kamiran & Calders (2012) reweighing. Each (group, label) cell is
    weighted by P(group)*P(label) / P(group, label), so the joint
    distribution of group x label becomes independent in the weighted
    sample.'''
    g = df_train[protected_col]
    y = df_train['TARGET']
    n = len(df_train)

    p_g = g.value_counts(normalize=True)
    p_y = y.value_counts(normalize=True)
    p_gy = pd.crosstab(g, y) / n

    w = pd.Series(index=df_train.index, dtype=float)
    for gv in p_g.index:
        for yv in p_y.index:
            mask = (g == gv) & (y == yv)
            w[mask] = p_g[gv] * p_y[yv] / p_gy.loc[gv, yv]
    return w.values


## M2 — Baseline: cleaning output only (no FE)

We choose LightGBM because it performs better than LogReg.

In [8]:
# Baseline features = raw columns from manifest, with the protected
# attributes folded back in (since baseline did not know about the
# protected/engineered/raw separation). We exclude CODE_GENDER (string)
# but keep its numeric proxies, the AGE_GROUP dummies, and AGE_YEARS.
baseline_features = (
    manifest['raw']
    + ['CODE_GENDER_M', 'AGE_YEARS']
    + [c for c in df.columns if c.startswith('AGE_GROUP_')]
)
# Sanity: all numeric, no string
baseline_features = [c for c in baseline_features
                     if c in df.columns and df[c].dtype != object]
print(f'Baseline features: {len(baseline_features)}')

baseline_results = run_experiment('baseline', baseline_features)
for r in baseline_results:
    print(f'  {r["model"]:<10s}  AUC={r["auc_overall"]:.4f}  '
          f'gender DP={r["gender_dp_gap"]:.4f}  EO={r["gender_eo_gap"]:.4f}  '
          f'age DP={r["age_dp_gap"]:.4f}  EO={r["age_eo_gap"]:.4f}')


Baseline features: 117
  LogReg      AUC=0.7423  gender DP=0.1892  EO=0.1611  age DP=0.4400  EO=0.4109
  LightGBM    AUC=0.7558  gender DP=0.1625  EO=0.1466  age DP=0.4119  EO=0.4064


## M3 — Experiment 1: `with_protected` (FE features + protected attrs)

Headline performance. The model sees every engineered feature plus the
protected attributes. This is the unconstrained upper bound on AUC and
serves as the fairness *lower bound* — if active mitigation is needed
anywhere, it's here.

In [9]:
# All non-string columns except IDs and the audit-only AGE_GROUP
exp_with_protected_features = [
    c for c in df.columns
    if c not in ['SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'AGE_GROUP']
       and df[c].dtype != object
]
print(f'with_protected features: {len(exp_with_protected_features)}')

with_protected_results = run_experiment('with_protected',
                                         exp_with_protected_features)
for r in with_protected_results:
    print(f'  {r["model"]:<10s}  AUC={r["auc_overall"]:.4f}  '
          f'gender DP={r["gender_dp_gap"]:.4f}  EO={r["gender_eo_gap"]:.4f}  '
          f'age DP={r["age_dp_gap"]:.4f}  EO={r["age_eo_gap"]:.4f}')


with_protected features: 211
  LogReg      AUC=0.7507  gender DP=0.1813  EO=0.1591  age DP=0.4342  EO=0.4081
  LightGBM    AUC=0.7675  gender DP=0.1642  EO=0.1538  age DP=0.4148  EO=0.4202


## M4 — Experiment 2: `without_protected` (FE features only, no protected)

The key fairness test. We strip the protected attributes
(`CODE_GENDER_M`, `AGE_YEARS`, all `AGE_GROUP_*` dummies) before training,
keeping every engineered and raw non-protected feature. EDA Finding 5
already showed gender is reconstructible at AUC 0.664, so we *expect* the
fairness gaps to barely move — confirming that "fairness by unawareness"
is insufficient on this dataset.

In [10]:
exp_without_protected_features = [
    c for c in exp_with_protected_features
    if c not in manifest['protected']
]
removed = set(exp_with_protected_features) - set(exp_without_protected_features)
print(f'without_protected features: {len(exp_without_protected_features)} '
      f'(removed {len(removed)} protected cols)')
print(f'Removed cols: {sorted(removed)}')

without_protected_results = run_experiment('without_protected',
                                            exp_without_protected_features)
for r in without_protected_results:
    print(f'  {r["model"]:<10s}  AUC={r["auc_overall"]:.4f}  '
          f'gender DP={r["gender_dp_gap"]:.4f}  EO={r["gender_eo_gap"]:.4f}  '
          f'age DP={r["age_dp_gap"]:.4f}  EO={r["age_eo_gap"]:.4f}')


without_protected features: 204 (removed 7 protected cols)
Removed cols: ['AGE_GROUP_25-35', 'AGE_GROUP_35-45', 'AGE_GROUP_45-55', 'AGE_GROUP_55+', 'AGE_GROUP_<25', 'AGE_YEARS', 'CODE_GENDER_M']
  LogReg      AUC=0.7465  gender DP=0.0674  EO=0.0411  age DP=0.5122  EO=0.4496
  LightGBM    AUC=0.7666  gender DP=0.1007  EO=0.0881  age DP=0.4190  EO=0.4605


In [11]:
def gap_comparison(results_a, results_b, label_a, label_b):
    print(f'{"Model":<10s} {"Metric":<18s}  {label_a:>11s}  {label_b:>11s}  {"Δ":>8s}')
    print('-' * 65)
    for ra, rb in zip(results_a, results_b):
        for metric in ['gender_dp_gap', 'gender_eo_gap', 'gender_auc_gap',
                       'age_dp_gap', 'age_eo_gap', 'age_auc_gap']:
            d = rb[metric] - ra[metric]
            arrow = '↑' if d > 1e-3 else ('↓' if d < -1e-3 else '~')
            print(f'  {ra["model"]:<8s} {metric:<18s}  {ra[metric]:>11.4f}  '
                  f'{rb[metric]:>11.4f}  {arrow}{abs(d):>7.4f}')
        print()

gap_comparison(with_protected_results, without_protected_results,
               'with_prot', 'without_prot')


Model      Metric                with_prot  without_prot         Δ
-----------------------------------------------------------------
  LogReg   gender_dp_gap            0.1813       0.0674  ↓ 0.1139
  LogReg   gender_eo_gap            0.1591       0.0411  ↓ 0.1181
  LogReg   gender_auc_gap           0.0023       0.0016  ~ 0.0007
  LogReg   age_dp_gap               0.4342       0.5122  ↑ 0.0780
  LogReg   age_eo_gap               0.4081       0.4496  ↑ 0.0415
  LogReg   age_auc_gap              0.0560       0.0624  ↑ 0.0064

  LightGBM gender_dp_gap            0.1642       0.1007  ↓ 0.0636
  LightGBM gender_eo_gap            0.1538       0.0881  ↓ 0.0658
  LightGBM gender_auc_gap           0.0015       0.0019  ~ 0.0004
  LightGBM age_dp_gap               0.4148       0.4190  ↑ 0.0042
  LightGBM age_eo_gap               0.4202       0.4605  ↑ 0.0403
  LightGBM age_auc_gap              0.0513       0.0528  ↑ 0.0015



## M5 — Repeated stratified split (seed robustness)

In [23]:
# Repeated stratified split across seeds for baseline vs mitigated LightGBM.
# Self-contained: resplits inside the loop, reuses the M1 helpers.
SEEDS = [0, 1, 2, 3, 4]

AUDIT_COLS = ['auc_overall',
              'gender_dp_gap', 'gender_eo_gap', 'gender_auc_gap',
              'age_dp_gap', 'age_eo_gap', 'age_auc_gap']

def _audit_lgbm_split(seed, mitigate):
    """One stratified resplit -> fit LightGBM -> audit. Returns an audit dict."""
    tr, te = train_test_split(df, test_size=0.20, stratify=df['TARGET'],
                              random_state=seed)
    X_tr, X_te = tr[exp_without_protected_features], te[exp_without_protected_features]
    y_tr = tr['TARGET']
    y_te = te['TARGET'].reset_index(drop=True)
    gender_te = te['CODE_GENDER'].reset_index(drop=True)
    age_te    = te['AGE_GROUP'].astype(str).reset_index(drop=True)

    sw = kamiran_calders_weights(tr, 'CODE_GENDER') if mitigate else None
    clf = fit_lgbm(X_tr, y_tr, sample_weight=sw)
    proba, pred = predict_lgbm(clf, X_te)
    if mitigate:
        pred = per_group_eo_threshold(y_te, proba, gender_te)
    return audit(y_te, proba, pred, gender_te, age_te)

rows = []
for seed in SEEDS:
    for label, mitigate in [('baseline_lgbm', False), ('mitigated_lgbm', True)]:
        a = _audit_lgbm_split(seed, mitigate)
        a.update({'config': label, 'seed': seed})
        rows.append(a)
        print(f'seed={seed}  {label:<14s}  AUC={a["auc_overall"]:.4f}  '
              f'gender EO={a["gender_eo_gap"]:.4f}  age EO={a["age_eo_gap"]:.4f}')

repeated_df = pd.DataFrame(rows)

seed=0  baseline_lgbm   AUC=0.7669  gender EO=0.0857  age EO=0.4168
seed=0  mitigated_lgbm  AUC=0.7662  gender EO=0.0068  age EO=0.4045
seed=1  baseline_lgbm   AUC=0.7643  gender EO=0.0808  age EO=0.3670
seed=1  mitigated_lgbm  AUC=0.7633  gender EO=0.0015  age EO=0.3467
seed=2  baseline_lgbm   AUC=0.7624  gender EO=0.0777  age EO=0.4103
seed=2  mitigated_lgbm  AUC=0.7598  gender EO=0.0020  age EO=0.3899
seed=3  baseline_lgbm   AUC=0.7692  gender EO=0.0793  age EO=0.4197
seed=3  mitigated_lgbm  AUC=0.7677  gender EO=0.0026  age EO=0.3649
seed=4  baseline_lgbm   AUC=0.7590  gender EO=0.0769  age EO=0.3914
seed=4  mitigated_lgbm  AUC=0.7584  gender EO=0.0092  age EO=0.3316


In [27]:
# Aggregate across seeds: mean +/- std for each config.
agg = (repeated_df
       .groupby('config')[AUDIT_COLS]
       .agg(['mean', 'std']))

# Pretty "mean +/- std" table
summary_repeated = pd.DataFrame(index=['baseline_lgbm', 'mitigated_lgbm'])
for col in AUDIT_COLS:
    summary_repeated[col] = [
        f'{agg.loc[cfg, (col, "mean")]:.4f} ± {agg.loc[cfg, (col, "std")]:.4f}'
        for cfg in summary_repeated.index
    ]
print(f'Repeated stratified split over seeds {SEEDS} (mean ± std, n={len(SEEDS)}):\n')
print(summary_repeated.T.to_string())

# Mitigation effect on the target metric (gender EO gap), per the spec
b_mean = agg.loc['baseline_lgbm',  ('gender_eo_gap', 'mean')]
m_mean = agg.loc['mitigated_lgbm', ('gender_eo_gap', 'mean')]
b_auc  = agg.loc['baseline_lgbm',  ('auc_overall',   'mean')]
m_auc  = agg.loc['mitigated_lgbm', ('auc_overall',   'mean')]
print(f'\nGender EO gap:  baseline {b_mean:.4f}  ->  mitigated {m_mean:.4f}  '
      f'(Δ={m_mean-b_mean:+.4f})')
print(f'Overall AUC:    baseline {b_auc:.4f}  ->  mitigated {m_auc:.4f}  '
      f'(Δ={m_auc-b_auc:+.4f}, the fairness cost)')

# Save for the report
summary_repeated.to_csv('repeated_split_results.csv')
print(f'\nSaved repeated_split_results.csv')

Repeated stratified split over seeds [0, 1, 2, 3, 4] (mean ± std, n=5):

                  baseline_lgbm   mitigated_lgbm
auc_overall     0.7644 ± 0.0040  0.7631 ± 0.0040
gender_dp_gap   0.0971 ± 0.0032  0.0250 ± 0.0071
gender_eo_gap   0.0801 ± 0.0035  0.0044 ± 0.0034
gender_auc_gap  0.0065 ± 0.0019  0.0067 ± 0.0034
age_dp_gap      0.3929 ± 0.0087  0.3639 ± 0.0078
age_eo_gap      0.4010 ± 0.0220  0.3675 ± 0.0300
age_auc_gap     0.0515 ± 0.0177  0.0549 ± 0.0178

Gender EO gap:  baseline 0.0801  ->  mitigated 0.0044  (Δ=-0.0757)
Overall AUC:    baseline 0.7644  ->  mitigated 0.7631  (Δ=-0.0013, the fairness cost)

Saved repeated_split_results.csv
